# 01. Agentic AI 기초: 단일 모델 호출에서 Agent까지

이 과정의 최종 목표는 사내 문서를 스스로 찾아 읽고 명확한 근거(Evidence)를 들어 답변하는 **운영형 RAG Agent**를 만드는 것입니다.
복잡해 보이는 RAG Agent도 결국 이 노트북에서 하나씩 다룰 **다섯 가지 핵심 부품(모델, 도구, 지침, 상태, 종료 조건)**으로 이루어져 있습니다.

첫 시간에는 사내 식당 메뉴와 통근 버스 조회처럼 작고 직관적인 예제를 통해 이 다섯 가지 부품을 하나씩 직접 조립해 봅니다.

## 학습 목표

- "누가 다음 행동을 결정하는가"를 기준으로 단순 모델 호출, 고정 workflow, Agent의 차이를 설명할 수 있습니다.
- 도구를 사용할 때 모델은 실행을 **요청**만 하고, 실제 파이썬 함수 **실행**은 우리 애플리케이션 코드가 맡는다는 사실을 출력으로 확인합니다.
- `create_deep_agent` 실행 결과에 담긴 메시지 목록(`messages`)을 한 줄씩 읽고 내부에서 어떤 왕복이 일어났는지 설명할 수 있습니다.
- 존재하지 않는 도구나 끝나지 않는 도구를 일부러 연결해 보고, Agent가 오류를 어떻게 전달받고 언제 멈추는지 확인합니다.

## 수업 시간표

| 절 | 내용 | 성격 | 분 |
|---|---|---|---:|
| 1 | 왜 Agent인가: 단일 호출, 고정 workflow, 다음 행동을 고르는 Agent | 설명 | 6 |
| 2 | 같은 질문 세 번 물어보기: 도구 없이, 도구 하나 묶기, 직접 실행 | 실습 | 9 |
| 3 | Agent 조립하기와 다섯 가지 구성요소 | 설명 + 실습 | 5 |
| 4 | 시스템 프롬프트(지침) 바꾸기 | 실습 | 5 |
| 5 | 메시지가 곧 상태다: 대화 기록을 한 줄씩 읽기 | 실습 | 11 |
| 6 | 종료 조건과 실패 다루기 (1): 존재하지 않는 도구 | 설명 + 실습 | 5 |
| 7 | 종료 조건과 실패 다루기 (2): 끝나지 않는 도구와 반복 한도 | 실습 | 6 |
| - | 직접 해보기: 내 업무 데이터로 세 단계 비교하기 | 내 것 | 10 |
| - | 확인 질문과 핵심 정리 | 설명 | 3 |


## 이 노트북의 진행 방식

Day01에서는 LangGraph의 `StateGraph`를 사용해 노드(node)와 엣지(edge)를 직접 이어 붙여 에이전트 루프를 만들었습니다. 이번 과정에서는 그 루프를 바닥부터 다시 짜지 않습니다. Deep Agents 라이브러리의 `create_deep_agent`가 **모델 호출 → 도구 실행 → 결과 전달 → 재호출**로 이어지는 에이전트 루프(harness)를 대신 조립해 주기 때문입니다. 우리는 조립된 에이전트에 어떤 부품을 넣고 실행 결과를 어떻게 읽어야 하는지에 집중합니다.

코드 셀 맨 윗줄의 표지는 해당 셀의 실습 방식을 뜻합니다.

- `# [따라하기]`: 코드를 있는 그대로 실행하고 출력 결과를 관찰합니다.
- `# [바꿔보기]`: 안내된 변수 값 하나를 직접 바꿔 봅니다. **실행하기 전에** 결과가 어떻게 달라질지 먼저 예상해 보세요.
- `# [깨뜨리기]`: 일부러 오류나 한도 초과를 일으켜 에이전트가 어디서 어떻게 멈추는지 관찰합니다.
- `# [내 것 만들기]`: 기본 예시가 채워져 있어 바로 실행할 수 있습니다. 동작을 확인한 뒤 내 업무 내용으로 값을 바꿔 다시 실행해 보세요.

모델 호출이 필요한 모든 셀은 **실제 Claude 모델**을 부릅니다. 모델이 어떤 도구를 고르고 어떻게 답할지는 실행할 때마다 조금씩 달라질 수 있으므로, 셀 아래의 **관찰 포인트**는 고정된 정답이 아니라 **방금 나온 출력을 읽는 기준**으로 삼아 주세요. 유료 호출 셀은 실행 직전에 `announce` 함수가 예상 호출 규모를 한 줄로 먼저 알려 줍니다.

In [ ]:
from rag_lab.settings import key_status, require_key

# 필수 API 키가 설정되어 있는지 먼저 확인합니다 (미설정 시 안내와 함께 멈춤)
require_key()
print(key_status())

`key_status()`는 실제 키 문자열을 노출하지 않고 `설정됨`/`미설정` 상태와 사용할 모델 이름, 추론 깊이(`effort`) 설정만 보여 줍니다. 만약 `미설정`으로 나와서 위 셀이 멈췄다면 `Day02_new/.env.example`을 참고해 `.env` 파일을 직접 채운 뒤 커널을 다시 시작해 주세요.

---

## 1절. 단일 호출, 고정 workflow, 다음 행동을 고르는 Agent (6분)

LLM을 활용해 사용자의 질문을 처리하는 방식은 크게 세 가지로 나뉩니다. 세 방식을 가르는 핵심 기준은 **"누가 다음 행동을 결정하는가"**입니다.

| 방식 | 다음 행동을 결정하는 주체 | 동작 예시 |
|---|---|---|
| **단일 모델 호출** | 다음 행동이 없음 (입력 1회 → 출력 1회) | 질문을 프롬프트로 보내고 답변 문장을 바로 받습니다 |
| **고정 workflow** | 개발자가 작성한 파이썬 코드가 순서를 정함 | 코드에서 요일로 메뉴 사전을 먼저 조회한 뒤, 그 결과를 프롬프트에 끼워 넣어 모델을 부릅니다 |
| **Agent** | 모델이 매 단계마다 상황을 보고 스스로 고름 | 메뉴 조회 도구를 부를지, 바로 답할지, 다른 도구를 추가로 부를지를 모델이 직접 판단합니다 |

**고정 workflow**는 실행 경로가 항상 일정해서 결과를 예측하기 쉽고, 문제가 생겼을 때 코드의 어느 줄에서 틀렸는지 바로 찾을 수 있습니다. 다만 처리해야 할 예외 상황과 분기가 늘어날수록 조건문 코드가 복잡해집니다.

반면 **Agent**는 분기 판단을 모델에게 맡겨 코드를 단순하게 유지할 수 있지만, 실행할 때마다 도구 호출 횟수나 순서가 달라질 수 있습니다. 따라서 어느 한쪽이 무조건 더 우월한 것이 아니라, **해결하려는 업무의 성격에 맞춰 알맞은 방식을 골라야 합니다.**

먼저 가장 단순한 **단일 모델 호출**부터 살펴봅시다. 아래 셀은 아무런 도구도 주지 않고 모델에게 이번 주 월요일 사내 식당 메뉴를 물어봅니다. 우리 회사의 이번 주 식단표는 모델이 사전 학습 데이터로 알 수 없는 내부 정보입니다.

> 참고: `make_model()`은 `Day02_new/.env`에 설정된 키와 모델 이름으로 Claude 호출 객체(`ChatAnthropic`)를 생성하는 도우미 함수입니다. 설정을 읽고 토큰 사용량을 기록하는 역할만 하며, 모델의 추론이나 응답 내용에는 아무런 영향을 주지 않습니다.


In [ ]:
# [따라하기] 유료, 모델 호출 약 1회
from rag_lab.settings import announce, make_model

question = "이번 주 월요일 사내 식당 메뉴가 뭐예요?"
announce("도구 없이 단일 호출로 묻기", model_calls=1)

model = make_model()
reply = model.invoke(question)

print("모델 답변:\n", reply.text)
print("\n도구 호출 요청(tool_calls):", reply.tool_calls)

**관찰할 포인트:**
- `tool_calls`가 빈 리스트(`[]`)인 것을 확인하세요. 연결된 도구가 없으므로 모델은 아무것도 조회하지 못하고 텍스트 답변만 돌려주었습니다.
- 답변 본문에서 모델이 사내 식당 메뉴를 **솔직히 모른다고 답하는지**, 아니면 그럴듯한 메뉴를 **지어내어(환각) 답하는지** 읽어 보세요.
- 답변 텍스트를 꺼낼 때 `reply.content` 대신 `reply.text`를 썼습니다. Claude처럼 확장 추론(thinking)을 지원하는 모델은 `reply.content` 안에 추론 블록과 텍스트 블록을 리스트로 담아 돌려주므로, 최종 답변 문자열만 꺼낼 때는 `.text` 속성을 쓰는 것이 편리합니다.

이번에는 질문 변수 `question`의 값 하나만 바꿔서 다시 실행해 봅시다. 실행하기 전에 아래 셀의 주석에 예상 결과를 먼저 생각해 보세요.

In [ ]:
# [바꿔보기] 유료, 모델 호출 약 1회
# 바꿀 값: question
# 먼저 예측해 보세요: 이번에도 모른다고 답할까요? 도구가 꼭 필요한 질문과 필요 없는 질문은 무엇이 다를까요?
question = "대한민국의 수도는 어디인가요?"
announce("도구 없이 묻기 (일반 상식 질문)", model_calls=1)

reply = make_model().invoke(question)
print("모델 답변:", reply.text)
print("도구 호출 요청(tool_calls):", reply.tool_calls)

위 표에서 정리한 **고정 workflow**와 **Agent**의 차이를 실제 LLM 호출 코드로 먼저 확인해 봅시다.

가장 흔한 고정 workflow는 **"파이썬 코드가 먼저 정해진 데이터(예: 식단표 `MENU`)를 꺼내 프롬프트에 넣은 뒤 모델을 한 번 호출하는 구조"**입니다. 식단 질문이 들어오면 잘 답하겠지만, 만약 사용자가 **통근 버스 시간**을 물어보면 어떻게 될까요? 아래 셀을 실행해 확인해 보세요.

In [ ]:
# [따라하기] 유료, 모델 호출 약 2회 — 조회할 데이터를 파이썬 코드가 고정해 둔 LLM workflow 예시

# [1단계] 파이썬 코드가 미리 들고 있는 사내 데이터입니다. 요일 -> 식당 메뉴, 노선 -> 출발 시각
MENU = {
    "월요일": "김치찌개",
    "화요일": "비빔밥",
    "수요일": "돈가스",
}
BUS = {
    "판교": "오후 6시",
    "수원": "오후 6시 30분",
}


def fixed_menu_llm_workflow(question: str) -> str:
    """질문 내용과 상관없이 무조건 식단표(MENU)만 프롬프트에 넣어 모델에게 답하게 하는 고정 workflow입니다.

    입력: question - 사용자의 질문 문자열
    출력: 모델이 식단표만 보고 만든 답변 문자열
    """
    # [2단계] 파이썬 코드가 항상 고정된 데이터(MENU)만 꺼내 프롬프트를 조립합니다
    instruction = "아래 사내 식단표 데이터만 보고 질문에 한 문장으로 답하세요."
    menu_part = f"[식단표 데이터]: {MENU}"
    question_part = f"[질문]: {question}"
    prompt = instruction + "\n" + menu_part + "\n" + question_part

    # [3단계] 모델은 도구를 고를 권한 없이 주어진 프롬프트로만 답변을 생성합니다
    reply_message = model.invoke(prompt)
    return reply_message.text


announce("고정 LLM workflow로 식단 질문과 통근 버스 질문을 차례대로 실행합니다", model_calls=2)
q_menu = "이번 주 월요일 사내 식당 메뉴가 뭐예요?"
q_bus = "월요일 판교행 통근 버스는 몇 시에 출발하나요?"

# 질문 1: 식단 질문 (MENU 안에 답이 있습니다)
menu_answer = fixed_menu_llm_workflow(q_menu)
print("[질문 1 — 식단 질문]:", q_menu)
print(" -> 고정 workflow 답변:", menu_answer)
print()

# 질문 2: 버스 질문 (BUS는 프롬프트에 들어가지 않습니다)
bus_answer = fixed_menu_llm_workflow(q_bus)
print("[질문 2 — 버스 질문]:", q_bus)
print(" -> 고정 workflow 답변:", bus_answer)

**관찰할 포인트:**
- `fixed_menu_llm_workflow`는 분명 내부에서 LLM(`model.invoke`)을 호출했지만, **무엇을 조회할지(`MENU`)를 파이썬 코드가 미리 고정**해 두었습니다.
- 질문 1(식단)의 답변에 위 `MENU`에 적어 둔 월요일 메뉴가 들어 있는지 확인하세요. 답이 식단표 안에 있는 질문이므로 고정 workflow가 잘 처리할 수 있는 경우입니다.
- 질문 2(통근 버스)에서는 우리 코드에 버스 시간표 `BUS`가 이미 정의되어 있는데도 프롬프트에는 `MENU`만 들어갔다는 점이 핵심입니다. 모델은 `BUS`를 볼 수 없으므로, 답변이 판교행 출발 시각을 말하는지 아니면 식단표에 버스 정보가 없다는 취지로 답하는지 읽어 보세요. 만약 시각을 말했다면 그 값이 식단표에서 나올 수 없는 정보이니 어디서 나왔는지 의심해 볼 만합니다.
- 물론 `if "버스" in question:` 같은 분기문을 사람이 끝없이 추가할 수도 있지만, 질문 표현이 다양해지거나 두 데이터를 동시에 봐야 하는 복합 질문이 들어오면 규칙이 금방 깨집니다. 이어지는 2~3절에서 **모델이 직접 도구를 골라 호출하는 Agent**가 이 한계를 어떻게 해결하는지 살펴봅시다.

---

## 2절. 같은 질문 세 번 물어보기: 도구 없이, 도구 하나 묶기, 직접 실행 (9분)

1절에서 고정 workflow는 "무엇을 조회할지"를 파이썬 코드가 정해 버린다는 점을 확인했습니다. 이제부터는 **같은 질문("이번 주 월요일 사내 식당 메뉴가 뭐예요?")을 세 가지 방식으로 연달아 물어보며** 모델이 조회 여부를 스스로 고르는 과정을 단계별로 봅니다.

1. **도구 없이 묻기:** 1절의 첫 코드 셀에서 이미 했습니다. 모델은 `tool_calls=[]`로 텍스트만 돌려주었습니다.
2. **도구 하나 묶기(`bind_tools`):** 아래 셀에서 모델에게 `get_menu` 도구 명세를 보여 주고, 모델이 어떤 요청서를 쓰는지 봅니다.
3. **직접 실행:** 그 요청서를 우리 코드가 꺼내 `get_menu` 함수를 실제로 실행합니다.

아래 셀은 도구 `get_menu`를 정의하고, 모델에 묶어 질문합니다. 도구가 조회하는 데이터는 1절에서 정의한 `MENU` 사전을 그대로 씁니다. 따라서 고정 workflow와 도구 방식은 **같은 데이터**를 쓰고, 달라지는 것은 "누가 조회를 결정하는가"뿐입니다.

In [ ]:
# [따라하기] 유료, 모델 호출 약 1회
from langchain_core.tools import tool


@tool
def get_menu(day: str) -> str:
    """요일(예: 월요일, 화요일)을 입력받아 그날의 사내 식당 메뉴를 돌려준다."""
    # 1절에서 정의한 MENU 사전에서 요일로 메뉴를 찾고, 없으면 안내 문구를 돌려줍니다
    not_found_message = "해당 요일의 메뉴 정보가 없습니다."
    return MENU.get(day, not_found_message)


question = "이번 주 월요일 사내 식당 메뉴가 뭐예요?"
announce("도구 하나를 묶어 호출하기", model_calls=1)

# [1단계] 모델에 get_menu 도구 명세를 묶습니다 (함수를 실행하는 것이 아니라 "이런 도구가 있다"고 알려 주는 것)
model_with_tool = make_model().bind_tools([get_menu])

# [2단계] 질문을 보내고, 모델이 쓴 요청서(tool_calls)를 그대로 받습니다
reply = model_with_tool.invoke(question)

print("텍스트 본문(text)          :", repr(reply.text))
print("도구 호출 요청(tool_calls) :", reply.tool_calls)

**관찰할 포인트:**
- `reply.tool_calls` 리스트 안에 도구 이름(`'name': 'get_menu'`)과 인자(`'args': {'day': '월요일'}`)가 들어 있는지 확인하세요.
- 출력 어디를 봐도 아직 **`김치찌개`라는 단어는 나타나지 않았습니다.** 모델은 도구를 불러 달라고 요청만 했을 뿐, `get_menu` 함수는 아직 한 번도 실행되지 않았기 때문입니다.
- 이제 아래 셀에서 모델이 보낸 요청서를 꺼내 우리 코드가 직접 함수를 실행해 봅시다.

In [ ]:
# [따라하기] 무료, 모델 호출 0회
# 바로 위 셀에서 받은 reply(모델의 응답)에 도구 호출 요청이 있을 때만 실행합니다
if reply.tool_calls:
    # [1단계] 모델이 쓴 요청서 중 첫 번째를 꺼냅니다
    first_request = reply.tool_calls[0]

    # [2단계] 우리 코드가 직접 get_menu 함수를 실행합니다 (모델 호출은 없습니다)
    tool_message = get_menu.invoke(first_request)

    # [3단계] 실행 결과가 담긴 ToolMessage를 확인합니다
    print("반환 객체 타입 :", type(tool_message).__name__)
    print("도구 실행 결과 :", tool_message.content)
else:
    print("이번에는 모델이 도구를 요청하지 않았습니다. 바로 위 셀을 다시 실행해 보세요.")

**관찰할 포인트:**
- 우리 파이썬 코드가 `get_menu.invoke(...)`를 호출하고 나서야 비로소 `김치찌개`라는 실행 결과가 담긴 `ToolMessage`가 만들어졌습니다.
- 만약 아주 드물게 `tool_calls`가 비어 있었다면 그것 역시 중요한 관찰입니다. 도구를 연결해 주더라도 실제로 그 도구를 쓸지 말지는 전적으로 모델이 판단합니다.

그렇다면 식당 메뉴와 전혀 관련 없는 질문을 던지면 모델은 어떻게 행동할까요? 먼저 결과를 예측해 본 뒤 아래 셀을 실행해 보세요.

In [ ]:
# [바꿔보기] 유료, 모델 호출 약 1회
# 바꿀 값: question
# 먼저 예측해 보세요: 통근 버스를 물어보면 모델이 get_menu 도구를 부를까요, 아니면 안 부를까요?
question = "통근 버스는 몇 시에 출발하나요?"
announce("도구 하나를 묶고 관련 없는 질문하기", model_calls=1)

# get_menu 도구만 묶은 모델에게 식당과 관련 없는 질문을 보냅니다
model_with_menu_tool = make_model().bind_tools([get_menu])
reply = model_with_menu_tool.invoke(question)

print("텍스트 본문(text)          :", repr(reply.text))
print("도구 호출 요청(tool_calls) :", reply.tool_calls)

**관찰할 포인트:**
- `tool_calls`가 비어 있다면, 모델이 `get_menu`의 설명("요일을 입력받아 그날의 사내 식당 메뉴를 돌려준다")을 읽고 통근 버스 질문에는 이 도구가 소용없다고 스스로 판단한 것입니다.
- 이처럼 도구를 쥐여 주었다고 해서 무조건 호출하는 것이 아니라, **상황에 따라 도구를 쓸지 말지 판단하는 것**이 바로 "모델이 다음 행동을 고른다"는 뜻입니다.

---

## 3절. Agent: 요청과 실행의 왕복을 Harness가 맡는다 (5분)

2절에서는 모델이 도구 요청을 내면 우리가 직접 `get_menu.invoke(...)`를 실행했습니다. 하지만 실제 대화에서는 그 실행 결과를 다시 모델에게 돌려주어 최종 답변 문장을 받아야 하고, 때로는 모델이 도구를 두세 번 연달아 부르기도 합니다. 이 왕복 루프를 매번 손으로 짜는 것은 번거롭습니다.

Deep Agents의 `create_deep_agent`는 바로 이 **모델 호출 ↔ 도구 실행 왕복을 자동으로 돌려주는 실행 틀(harness)**입니다. 우리는 모델과 도구 목록만 넘겨 에이전트를 만들고 질문을 넣기만 하면 됩니다.

In [ ]:
# [따라하기] 유료, 모델 호출 약 2회
from deepagents import create_deep_agent

question = "이번 주 월요일 사내 식당 메뉴가 뭐예요?"
announce("Agent 첫 실행", model_calls=2)

# [1단계] 모델과 도구 목록만 넘겨 Agent를 만듭니다 (모델 호출 <-> 도구 실행 왕복은 harness가 맡습니다)
agent = create_deep_agent(model=make_model(), tools=[get_menu])

# [2단계] 사용자 질문을 messages 형식으로 넣어 실행합니다
user_message = {"role": "user", "content": question}
result = agent.invoke({"messages": [user_message]})

# [3단계] 마지막 메시지가 최종 답변이고, messages 전체 길이가 오간 메시지 수입니다
final_message = result["messages"][-1]
print("최종 답변:\n", final_message.text)
print("\n오간 전체 메시지 수:", len(result["messages"]))

**관찰할 포인트:**
- 우리가 직접 `get_menu.invoke(...)`를 호출하는 코드를 쓰지 않았는데도, 마지막 메시지(`result["messages"][-1].text`)에 월요일 메뉴인 **김치찌개**가 자연스러운 문장으로 들어가 있습니다.
- `오간 전체 메시지 수`가 4개(사용자 질문 → 모델의 도구 요청 → 도구 실행 결과 → 모델의 최종 답변) 이상으로 찍힌 것을 보세요. 모델이 도구를 추가로 더 불렀다면 메시지 수가 그만큼 더 늘어납니다.

### Agent를 이루는 다섯 가지 구성요소

어떤 프레임워크를 쓰든 Agent는 다음 다섯 가지 부품으로 설명할 수 있습니다.
1. **모델(Model):** 현재 상태를 읽고 다음 행동을 판단하는 두뇌
2. **도구(Tools):** 외부 데이터를 조회하거나 액션을 수행하는 손과 발
3. **지침(Instructions):** 모델이 매번 참고하는 역할 부여와 행동 규칙(시스템 프롬프트, Skills 등)
4. **상태(State):** 지금까지 오간 대화와 도구 실행 결과를 담아 두는 기억 장소
5. **종료 조건(Stopping Condition):** 모델이 스스로 답변을 마쳤거나, 반복 한도(`recursion_limit`)에 도달했을 때 멈추는 기준

아래 셀을 실행해 `create_deep_agent` 함수의 인자들이 이 다섯 가지 구성요소와 어떻게 일대일로 대응하는지 확인해 봅시다.

In [ ]:
# [따라하기] 무료, 모델 호출 0회
import inspect

# create_deep_agent가 받는 인자 이름들을 읽어 옵니다
params = inspect.signature(create_deep_agent).parameters

# 구성요소 -> 그에 해당하는 create_deep_agent 인자 이름들
roles = {
    "1. 모델 (판단 주체)  ": ["model"],
    "2. 도구 (외부 연결)  ": ["tools", "subagents"],
    "3. 지침 (행동 규칙)  ": ["system_prompt", "skills", "memory"],
    "4. 상태 (기억과 저장)": ["state_schema", "backend", "checkpointer", "store"],
}

for role, arg_names in roles.items():
    # 현재 설치된 버전에 없는 인자에는 "(현재 버전에 없음)"을 붙입니다
    formatted = []
    for name in arg_names:
        if name in params:
            formatted.append(name)
        else:
            formatted.append(name + "(현재 버전에 없음)")
    print(f"{role} -> {', '.join(formatted)}")

# 종료 조건(recursion_limit)은 생성 인자에 있는지 따로 확인합니다
has_recursion_limit = "recursion_limit" in params
print("5. 종료 조건         -> recursion_limit이 생성 인자에 있는가?", has_recursion_limit)

**관찰할 포인트:**
- **모델, 도구, 지침, 상태**와 관련된 설정은 모두 `create_deep_agent(...)`로 에이전트를 **만들 때** 인자로 넘깁니다.
- 반면 다섯 번째 부품인 **종료 조건(`recursion_limit`)**만 생성 인자 목록에 없습니다(`False`). 반복 상한은 에이전트를 만들 때가 아니라 `agent.invoke(..., config={"recursion_limit": N})`처럼 **실행할 때** 지정하기 때문입니다. 이 동작은 7절에서 직접 확인합니다.

---

## 4절. 시스템 프롬프트(지침) 바꾸기 (5분)

지침(`system_prompt`)은 모델이 매번 호출될 때마다 가장 먼저 읽는 **역할 설명서**입니다. 동일한 도구(`get_menu`)와 동일한 질문을 주었을 때, `system_prompt`만 바꾸면 무엇이 달라지고 무엇이 그대로 유지되는지 살펴봅시다.

In [ ]:
# [따라하기] 유료, 모델 호출 약 2회
announce("시스템 프롬프트: 한 문장으로 답하기", model_calls=2)

# 도구와 질문은 3절과 같게 두고 system_prompt(지침)만 추가합니다
one_sentence_prompt = "당신은 사내 생활 안내원입니다. 군더더기 없이 딱 한 문장으로만 답합니다."
agent = create_deep_agent(
    model=make_model(),
    tools=[get_menu],
    system_prompt=one_sentence_prompt,
)

user_message = {"role": "user", "content": question}
result = agent.invoke({"messages": [user_message]})
print(result["messages"][-1].text)

**관찰할 포인트:**
- 3절에서 별도 지침 없이 실행했을 때의 답변과 문장 길이·말투를 비교해 보세요.
- 우리가 넘긴 `system_prompt`는 Deep Agents 프레임워크가 기본으로 넣어 주는 시스템 지침 뒤에 덧붙여집니다(실제로 모델에게 전달되는 전체 시스템 프롬프트 원문은 03번 노트북에서 직접 찍어 확인합니다).

이번에는 시스템 프롬프트의 지침을 "한 문장 답변"에서 **"마크다운 표 형식 답변"**으로 바꿔 보겠습니다. 실행하기 전에 답변 모양과 도구 호출 횟수(메시지 수) 중 무엇이 바뀌고 무엇이 그대로일지 먼저 예상해 보세요.


In [ ]:
# [바꿔보기] 유료, 모델 호출 약 2회
# 바꿀 값: system_prompt
# 먼저 예측해 보세요: 답변의 출력 형식과 도구 호출 횟수(메시지 수) 중 무엇이 바뀌고 무엇이 그대로일까요?
announce("시스템 프롬프트: 마크다운 표로 답하기", model_calls=2)

# 지침만 "마크다운 표 형식"으로 바꾸고, 도구와 질문은 그대로 둡니다
table_prompt = "당신은 사내 생활 안내원입니다. 요일과 메뉴를 마크다운 표(| 요일 | 메뉴 |) 형식으로 정리해 답합니다."
agent = create_deep_agent(
    model=make_model(),
    tools=[get_menu],
    system_prompt=table_prompt,
)

user_message = {"role": "user", "content": question}
result = agent.invoke({"messages": [user_message]})
print(result["messages"][-1].text)
print("\n오간 전체 메시지 수:", len(result["messages"]))

**관찰할 포인트:**
- 최종 답변의 **출력 형식**은 마크다운 표로 바뀌었지만, 메뉴를 알아내기 위해 `get_menu` 도구를 한 번 호출하고 결과를 받아오는 **메시지 흐름**이 앞 셀과 같은지 `오간 전체 메시지 수`로 비교해 보세요. 앞 셀(3절)이 4개였다면 이번에도 4개일 가능성이 높지만, 모델이 도구를 더 부르면 달라질 수 있습니다. 어느 쪽이든 형식 지침 때문에 늘었는지, 모델의 도구 선택이 달라져서 늘었는지를 `show(...)`로 구분해 읽는 것이 목적입니다.
- 이처럼 출력 형식에 관한 지침은 주로 마지막 답변을 작성하는 방식을 바꾸며, 도구를 부르는 과정 자체는 질문과 도구 명세에 의해 결정되는 경우가 많습니다.

---

## 5절. 메시지가 곧 상태(State)다 (11분)

Agent의 "상태(State)"란 거창한 것이 아니라 **지금까지 주고받은 메시지 목록(`result["messages"]`)** 그 자체입니다.
1. 사용자의 질문(`HumanMessage`)
2. 모델의 도구 호출 요청(`AIMessage`, `tool_calls` 포함)
3. 도구의 실행 결과(`ToolMessage`)
4. 모델의 최종 답변(`AIMessage`)

이 메시지들이 리스트에 순서대로 차곡차곡 쌓이고, 모델은 매 단계마다 **지금까지 쌓인 메시지 목록 전체**를 처음부터 읽으며 다음 행동을 결정합니다. 따라서 이 메시지 목록만 읽을 줄 알면 에이전트가 내부에서 무슨 일을 했는지 완벽히 파악할 수 있습니다.

먼저 간단한 `for` 반복문으로 방금 실행한 `result["messages"]`를 한 줄씩 출력해 봅시다.

In [ ]:
# [따라하기] 무료, 모델 호출 0회
# 바로 앞 셀의 result["messages"]를 메시지 한 줄씩 읽어 출력합니다
for msg in result["messages"]:
    msg_type = type(msg).__name__                      # HumanMessage, AIMessage, ToolMessage 등
    tool_calls = getattr(msg, "tool_calls", None)      # 도구 요청이 없는 메시지는 None
    preview = msg.text[:60].replace("\n", " ")         # 본문은 앞 60자만, 줄바꿈은 공백으로
    print(f"{msg_type:13s} | tool_calls={tool_calls} | text={preview!r}")

**관찰할 포인트:**
- 일반적으로 `HumanMessage` → `AIMessage`(도구 요청) → `ToolMessage`(도구 결과) → `AIMessage`(최종 답변) 순서로 찍힙니다.
- 도구를 요청하는 첫 번째 `AIMessage`는 `text`가 비어 있는 대신 `tool_calls`에 호출할 함수 이름과 인자가 들어 있습니다.
- 이어지는 `ToolMessage`의 `text`에는 파이썬 함수가 반환한 값(`김치찌개`)이 들어 있고, 마지막 `AIMessage`가 이를 바탕으로 작성한 최종 답변입니다.

이번에는 통근 버스 출발 시각을 알려 주는 두 번째 도구(`get_bus_time`)를 추가하고, 한 질문에서 **식당 메뉴와 통근 버스 시각 두 가지를 동시에** 물어봅시다.

In [ ]:
# [바꿔보기] 유료, 모델 호출 약 2~3회
# 바꿀 값: tools 목록에 get_bus_time 추가 및 복합 질문 전달
# 먼저 예측해 보세요: 메시지 줄이 몇 개로 늘어날까요? 한 AIMessage 안에 두 도구 요청이 동시에 들어갈까요, 아니면 차례대로 부를까요?
@tool
def get_bus_time(route: str) -> str:
    """통근 버스 노선 이름(예: 판교, 수원)을 받아 출발 시각을 돌려준다."""
    # 1절에서 정의한 BUS 사전에서 노선으로 출발 시각을 찾고, 없으면 안내 문구를 돌려줍니다
    not_found_message = "해당 노선 정보가 없습니다."
    return BUS.get(route, not_found_message)


announce("도구 두 개, 질문 두 가지", model_calls=3)

# [1단계] 도구 두 개를 가진 Agent를 만듭니다
agent = create_deep_agent(model=make_model(), tools=[get_menu, get_bus_time])

# [2단계] 한 질문에 식당 메뉴와 버스 시각을 함께 묻습니다
combined_question = "월요일 식당 메뉴와 판교행 통근 버스 출발 시각을 알려 주세요."
user_message = {"role": "user", "content": combined_question}
result = agent.invoke({"messages": [user_message]})

# [3단계] 메시지를 한 줄씩 출력합니다 (바로 아래에서 정의할 show 함수의 원형입니다)
for msg in result["messages"]:
    msg_type = type(msg).__name__
    tool_calls = getattr(msg, "tool_calls", None)
    preview = msg.text[:60].replace("\n", " ")
    print(f"{msg_type:13s} | tool_calls={tool_calls} | text={preview!r}")

**관찰할 포인트:**
- 첫 번째 `AIMessage`의 `tool_calls` 안에 `get_menu`와 `get_bus_time` 두 개가 나란히 들어 있다면 모델이 **병렬 도구 호출(parallel tool calling)**을 선택한 것입니다. 반대로 하나를 먼저 부르고 결과를 본 뒤 다음 도구를 불렀다면 **순차 호출**을 한 것입니다. 두 방식 모두 정상이며, 모델이 상황에 따라 선택합니다.

매번 긴 `tool_calls` 딕셔너리를 그대로 출력하면 한눈에 읽기 불편합니다. 앞으로 메시지 흐름을 깔끔하게 볼 수 있도록 도구 실행 성공/실패 상태(`status`)까지 함께 보여 주는 작은 출력 함수 `show(messages)`를 정의해 두겠습니다.

In [ ]:
# [따라하기] 무료, 모델 호출 0회
def show(messages):
    """메시지 목록을 번호, 종류, 도구 상태, 도구 호출 내역, 본문 미리보기로 한 줄씩 출력한다.

    입력: messages - agent.invoke 결과의 result["messages"] 같은 메시지 리스트
    출력: 반환값은 없고, 메시지마다 한 줄씩 화면에 출력한다.
    """
    for idx, msg in enumerate(messages):
        # [1단계] 이 메시지가 요청한 도구 호출 목록을 "이름(인자)" 문자열로 만듭니다 (없으면 "-")
        raw_calls = getattr(msg, "tool_calls", None) or []
        call_texts = []
        for call in raw_calls:
            call_texts.append(f"{call['name']}({call['args']})")
        calls_str = ", ".join(call_texts) or "-"

        # [2단계] ToolMessage에만 있는 실행 상태(success/error)를 꺼냅니다 (없으면 "-")
        status = getattr(msg, "status", "-")

        # [3단계] 본문은 앞 50자만, 줄바꿈은 공백으로 바꿔 한 줄로 만듭니다
        preview = msg.text[:50].replace("\n", " ")

        print(f"{idx:2d} | {type(msg).__name__:13s} | status={status:7s} | 도구요청={calls_str} | {preview!r}")


show(result["messages"])

**관찰할 포인트:**
- 바로 앞 셀의 출력과 내용은 완전히 같지만, 몇 번째 메시지에서 어떤 도구가 호출되었고 결과 상태(`status`)가 무엇인지 훨씬 읽기 쉬워졌습니다.
- `status` 칸은 `ToolMessage`에서만 값을 가집니다. 도구가 정상 실행되었으면 `success`, 오류가 났으면 `error`가 찍힙니다. 이어지는 6절에서 `status=error`가 찍히는 상황을 실제 모델 호출로 확인해 봅니다.

---

## 6절. 종료 조건과 실패 다루기 (1): 존재하지 않는 도구 (5분)

Agent는 기본적으로 모델이 더 이상 도구를 부르지 않고 최종 답변을 내놓을 때 멈춥니다. 하지만 실무에서는 세 가지 흔한 실패 상황이 발생합니다.
1. 모델이 **실행 노드에 등록되지 않은 도구**를 호출하려고 시도하는 경우
2. 도구가 기대한 결과를 주지 않아 모델이 **끝없이 도구를 다시 호출**하는 경우
3. 가장 조용하고 위험한 실패로, **도구를 써야 하는 질문인데도 도구를 부르지 않고 그럴듯하게 지어내어 답하고 정상 종료**하는 경우

먼저 첫 번째 상황을 살펴봅시다. 만약 모델의 도구 스키마에는 `get_price`가 보이지만 정작 에이전트의 도구 실행 노드(`tools=[get_menu]`)에는 등록되어 있지 않다면, 모델이 `get_price`를 호출했을 때 에이전트 전체가 에러로 죽을까요? 아래 셀을 실행해 실제 모델 호출로 확인해 봅시다.

In [ ]:
# [깨뜨리기] 유료, 모델 호출 약 2회 — 실행 노드에 등록되지 않은 도구(get_price)를 모델이 호출했을 때의 복구 흐름 확인
from langchain.agents.middleware import wrap_model_call


@tool
def get_price(item: str) -> str:
    """음식 메뉴 이름을 받아 가격을 조회한다."""
    return "5,000원"


# [1단계] 모델 시야(스키마)에만 get_price를 노출하고, 실제 실행 도구 목록(tools)에는 get_menu만 등록합니다
@wrap_model_call
def expose_unregistered_tool(request, handler):
    """모델을 호출하기 직전에 도구 목록 끝에 get_price를 몰래 덧붙인다.

    입력: request - 모델 호출 요청, handler - 다음 단계를 실행하는 함수
    출력: 도구 목록이 바뀐 요청으로 handler를 실행한 결과
    """
    visible_tools = [*request.tools, get_price]
    return handler(request.override(tools=visible_tools))


announce("실행 노드에 없는 도구(get_price)를 모델이 호출할 때의 ToolMessage 확인", model_calls=2)
mismatch_agent = create_deep_agent(
    model=make_model(),
    tools=[get_menu],
    middleware=[expose_unregistered_tool],
)

# [2단계] 실제 모델에게 김치찌개 가격을 물어 get_price 호출을 유도합니다
price_question = {"role": "user", "content": "김치찌개 가격이 얼마예요?"}
mismatch_out = mismatch_agent.invoke({"messages": [price_question]})
show(mismatch_out["messages"])

# [3단계] status가 error인 ToolMessage를 모아 첫 번째 오류 원문을 출력합니다
tool_errors = []
for m in mismatch_out["messages"]:
    if m.type == "tool" and m.status == "error":
        tool_errors.append(m)

if tool_errors:
    print("\n도구 노드가 반환한 오류 메시지 원문:")
    print(tool_errors[0].content)

**관찰할 포인트:**
- `status`가 `error`인 `ToolMessage`가 있는지 찾아보세요(바로 아래에 원문을 모아 출력합니다). 그 본문에는 `get_price is not a valid tool, try one of [...]`처럼 **사용 가능한 도구 이름 목록**까지 친절하게 안내되어 있습니다. 목록에는 우리가 등록한 `get_menu`뿐 아니라 Deep Agents가 기본으로 넣어 주는 파일 도구(`ls`, `read_file`, `grep` 등)도 함께 나옵니다.
- 덕분에 프로그램이 예외로 죽지 않고, 다음 턴에서 모델이 이 오류 메시지를 읽어 다른 도구를 고르거나 사용자에게 안내할 수 있습니다.

그렇다면 실제 모델에게 프롬프트로 "없는 도구(`get_price`)를 써서 가격을 찾아 달라"고 시키면 어떻게 행동할까요? 아래 셀을 실행해 확인해 봅시다.

In [ ]:
# [깨뜨리기] 유료, 모델 호출 약 2회
announce("없는 도구(get_price)를 쓰라고 지시하기", model_calls=2)

# 이번에는 middleware 없이, get_menu만 가진 평범한 Agent에게 없는 도구를 쓰라고 시킵니다
agent = create_deep_agent(model=make_model(), tools=[get_menu])
order_message = {"role": "user", "content": "get_price 도구로 김치찌개 가격을 찾아 주세요."}
result = agent.invoke({"messages": [order_message]})

show(result["messages"])

# 모델이 실제로 요청한 도구 이름을 메시지마다 모아서 한 리스트로 만듭니다
asked_tools = []
for msg in result["messages"]:
    calls = getattr(msg, "tool_calls", None) or []
    for call in calls:
        asked_tools.append(call["name"])

print("\n모델이 실제로 요청한 도구 목록 :", asked_tools)
print("get_price를 요청했는가?        :", "get_price" in asked_tools)

**관찰할 포인트:**
- 모델이 `get_price`를 실제로 요청했는지 확인해 보세요. 대부분의 경우 모델은 자신에게 주어진 도구 명세에 `get_price`가 없다는 사실을 알고 아예 호출하지 않거나, 기본 제공된 파일 탐색 도구(`ls`, `glob` 등)로 가격 정보가 있는지 찾아본 뒤 "`get_price` 도구가 없어 가격을 확인할 수 없다"고 답합니다.
- 이처럼 파이썬 예외가 발생하지 않고 정상적으로 텍스트 답변을 내놓고 종료했다고 해서 늘 사용자가 원한 작업이 성공한 것은 아니라는 점을 기억해 두세요.

---

## 7절. 종료 조건과 실패 다루기 (2): 끝나지 않는 도구와 반복 한도 (6분)

두 번째 실패 유형은 도구가 항상 **"아직 작업이 안 끝났으니 다시 호출하세요"**라고만 답해 모델이 끝없이 재호출하는 무한 루프 상황입니다.

이때 에이전트를 강제로 멈춰 세우는 안전장치가 실행 설정의 `recursion_limit`입니다. 그래프 단계 수가 이 한도를 넘어가면 `GraphRecursionError` 예외가 발생하며 실행이 중단됩니다. 아래 셀에서는 `runs` 리스트에 기록을 남겨 도구가 실제로 몇 번 실행되었는지 세어 봅니다.


In [ ]:
# [깨뜨리기] 유료, 모델 호출 약 3회
from langgraph.errors import GraphRecursionError

# 도구가 실제로 몇 번 실행되었는지 세기 위한 기록 리스트입니다
runs = []


@tool
def poll_status(job: str) -> str:
    """작업 상태를 확인한다. 작업이 완료될 때까지 같은 인자로 다시 호출해야 한다."""
    runs.append(job)
    return "상태: 대기 중. 아직 완료되지 않았습니다. 완료될 때까지 다시 호출하세요."


announce("끝나지 않는 도구 (recursion_limit=6)", model_calls=3)
agent = create_deep_agent(model=make_model(), tools=[poll_status])

polling_request = {"role": "user", "content": "작업 job-1이 완료될 때까지 poll_status로 확인하고 알려 주세요."}

# 한도에 걸리면 GraphRecursionError가 나는 것이 이 실습에서 관찰하려는 정상 동작입니다
try:
    out = agent.invoke(
        {"messages": [polling_request]},
        config={"recursion_limit": 6},
    )
    final_text = out["messages"][-1].text[:80]
    print("한도 안에서 스스로 멈췄습니다. 최종 답:", final_text)
except GraphRecursionError as error:
    error_text = str(error)[:80]
    print("반복 한도 도달 ->", type(error).__name__, "|", error_text)

# runs에는 poll_status가 실제로 실행될 때마다 기록이 쌓입니다
print("실제 도구 실행 횟수:", len(runs))

**관찰할 포인트:**
- `GraphRecursionError`가 출력되었다면, 모델이 도구의 지시대로 계속 재호출을 시도하다가 `recursion_limit=6`에 걸려 안전하게 강제 종료된 것입니다(간혹 모델이 두세 번 호출해 보고 "계속 대기 중입니다"라며 스스로 멈추기도 합니다).
- 만약 반복 한도를 지정하지 않으면 기본 한도까지 계속 모델을 호출하며 큰 비용과 시간이 낭비될 수 있습니다. 설치된 deepagents 0.7.32는 `create_deep_agent`가 만든 그래프에 `recursion_limit`의 기본값으로 9,999단계를 설정해 두었습니다(`deepagents/graph.py:1023`). 즉 한도를 직접 줄여 두지 않으면 사실상 아주 많은 단계까지 계속 돌 수 있습니다.

이번에는 `recursion_limit`을 `6`에서 `4`로 줄여서 다시 실행해 봅니다. 실행 전에 도구 실행 횟수(`len(runs)`)가 몇 번으로 줄어들지 먼저 예상해 보세요.

In [ ]:
# [바꿔보기] 유료, 모델 호출 약 2회
# 바꿀 값: recursion_limit (6에서 4로 변경)
# 먼저 예측해 보세요: recursion_limit이 4면 도구 실행 횟수(len(runs))는 몇 번일까요? 두 숫자가 같을까요?
# 앞 셀에서 쌓인 실행 기록을 비우고 다시 셉니다
runs.clear()
announce("끝나지 않는 도구 (recursion_limit=4)", model_calls=2)
agent = create_deep_agent(model=make_model(), tools=[poll_status])

polling_request = {"role": "user", "content": "작업 job-1이 완료될 때까지 poll_status로 확인하고 알려 주세요."}

# 한도에 걸리면 GraphRecursionError가 나는 것이 이 실습에서 관찰하려는 정상 동작입니다
try:
    out = agent.invoke(
        {"messages": [polling_request]},
        config={"recursion_limit": 4},
    )
    final_text = out["messages"][-1].text[:80]
    print("한도 안에서 스스로 멈췄습니다. 최종 답:", final_text)
except GraphRecursionError as error:
    error_text = str(error)[:80]
    print("반복 한도 도달 ->", type(error).__name__, "|", error_text)

# runs에는 poll_status가 실제로 실행될 때마다 기록이 쌓입니다
print("실제 도구 실행 횟수:", len(runs))

**관찰할 포인트:**
- `recursion_limit`을 6에서 4로 줄였을 때 도구 실행 횟수(`len(runs)`)가 앞 셀보다 같거나 줄어드는지 비교해 보세요. 한도가 작을수록 도구를 실행할 수 있는 단계가 적어지므로 보통 줄어듭니다(한 번의 실행에서는 6일 때 2번, 4일 때 1번이 나올 수 있습니다).
- 다만 **`recursion_limit` 숫자(4)와 실제 도구 실행 횟수(1번 또는 2번)가 1:1로 같지는 않다**는 점에 주목하세요. 에이전트 그래프 안에서 입력 처리, 모델 노드 실행, 도구 노드 실행이 각각 한 단계(step)씩 소비하기 때문입니다.
- 바로 이 때문에 05번 노트북에서는 그래프 단계 수(`recursion_limit`)에만 의존하지 않고, **모델 호출 횟수·도구 호출 횟수·검색 횟수를 직접 세는 전용 예산(`Budget`) 객체**를 따로 만들어 관리합니다.

---

## 직접 해보기 (10분)

지금까지 배운 세 단계(**1. 도구 없이 단일 호출 → 2. 도구 하나 묶어 요청 확인 → 3. Agent로 왕복 실행**)를 내 업무 주제로 직접 돌려 봅니다.

아래 두 셀에는 임직원 잔여 연차 조회 예시가 미리 채워져 있어 그대로 실행할 수 있습니다. 먼저 한 번 실행해 동작을 본 뒤, `MY_DATA`, `MY_QUESTION`, 그리고 `my_lookup` 함수의 독스트링(설명)과 인자 이름을 본인의 업무 데이터(예: 서버 상태 조회, 부서별 담당자 조회, 프로젝트 코드 조회 등)로 바꿔 다시 실행해 보세요.

In [ ]:
# [내 것 만들기] 유료, 모델 호출 약 4회
# 예시 데이터입니다. 내 업무의 간단한 사전(키: 값)과 질문으로 자유롭게 바꿔 보세요.
MY_DATA = {"김민수": "12일", "이서연": "7일", "박지훈": "3일"}
MY_QUESTION = "김민수 님의 남은 연차는 며칠인가요?"


@tool
def my_lookup(name: str) -> str:
    """직원 이름을 입력받아 그 사람의 남은 연차 일수를 돌려준다."""  # 내 업무에 맞게 설명과 인자 이름을 바꾸세요
    not_found_message = "해당 이름의 연차 정보가 없습니다."
    return MY_DATA.get(name, not_found_message)


if not MY_QUESTION.strip() or not MY_DATA:
    print("MY_DATA와 MY_QUESTION이 비어 있습니다. 조회용 데이터 몇 개와 질문 문장을 채운 뒤 다시 실행하세요.")
else:
    announce("내 업무 데이터로 세 단계 비교", model_calls=4)

    # 1단계: 도구 없이 모델에게 바로 묻기
    plain = make_model().invoke(MY_QUESTION)
    plain_preview = plain.text[:100].replace("\n", " ")
    print("1) 도구 없이 단일 호출 :", plain_preview)

    # 2단계: 도구 하나를 묶어 호출 요청(tool_calls)만 받아 보기
    model_with_my_tool = make_model().bind_tools([my_lookup])
    asked = model_with_my_tool.invoke(MY_QUESTION)
    print("2) 도구 하나 요청 확인 :", asked.tool_calls)

    # 3단계: Agent로 도구 실행과 최종 답변까지 한 번에 돌리기
    mine = create_deep_agent(model=make_model(), tools=[my_lookup])
    my_message = {"role": "user", "content": MY_QUESTION}
    mine_result = mine.invoke({"messages": [my_message]})
    agent_preview = mine_result["messages"][-1].text[:100].replace("\n", " ")
    print("3) Agent 최종 답변     :", agent_preview)

**관찰할 포인트:**
- 세 단계의 출력을 나란히 비교해 보세요.
  1. 도구 없이 물었을 때 모델이 사내 정보를 모른다고 답했나요?
  2. 도구를 묶었을 때 `tool_calls`에 내가 정의한 함수 이름과 인자가 정확히 들어갔나요?
  3. Agent가 도구가 돌려준 값을 받아 최종 답변 문장을 완성했나요?
- 만약 내 업무 지식이 `{이름: 값}` 같은 단순 파이썬 사전 하나로 딱 떨어지지 않고 긴 규정 문서나 여러 표에 흩어져 있다면, 바로 그 지점이 04번 노트북부터 다룰 **검색(Retrieval) 설계**가 필요한 이유입니다.

이어서 아래 셀에서 내 업무 Agent의 말투와 답변 규칙을 정하는 `MY_SYSTEM_PROMPT`를 바꿔 실행하고, `show(...)`로 전체 메시지 기록을 확인해 보세요.

In [ ]:
# [내 것 만들기] 유료, 모델 호출 약 2회
# 내 업무 역할에 맞는 말투와 답변 형식 규칙으로 자유롭게 바꿔 보세요.
MY_SYSTEM_PROMPT = "당신은 인사팀 안내 담당자입니다. 조회한 숫자를 정확히 포함해 존댓말 한 문장으로 답합니다."

if not MY_SYSTEM_PROMPT.strip():
    print("MY_SYSTEM_PROMPT가 비어 있습니다. 내 업무에 맞는 역할과 답변 규칙을 적고 다시 실행하세요.")
else:
    announce("내 시스템 프롬프트 적용하기", model_calls=2)

    # 내 도구와 내 지침(system_prompt)을 함께 넘겨 Agent를 만듭니다
    mine = create_deep_agent(
        model=make_model(),
        tools=[my_lookup],
        system_prompt=MY_SYSTEM_PROMPT,
    )
    my_message = {"role": "user", "content": MY_QUESTION}
    mine_result = mine.invoke({"messages": [my_message]})

    # show 함수로 전체 메시지 기록을 한 줄씩 확인합니다
    show(mine_result["messages"])

**관찰할 포인트:**
- 마지막 `AIMessage`의 답변 말투가 내가 작성한 `MY_SYSTEM_PROMPT`를 잘 따랐는지, 그리고 앞선 도구 호출 흐름과 메시지 개수가 어떻게 유지되었는지 확인해 보세요.

---

## 확인 질문

1. 같은 사내 식당 메뉴 질문에 대해 "도구 없이 단일 호출", "도구 하나 묶기(`bind_tools`)", "Agent 실행(`create_deep_agent`)"은 각각 무엇을 돌려주었나요? 실제 메뉴 이름(`김치찌개`)은 어느 단계에서 처음으로 나타났나요?
2. 도구를 묶은 모델 호출에서 `get_menu` 파이썬 함수를 실제로 실행한 주체는 모델인가요, 아니면 우리 애플리케이션 코드인가요?
3. 4절에서 시스템 프롬프트를 바꿨을 때 달라진 부분과 달라지지 않은 부분을 메시지 목록(`messages`) 기준으로 설명해 보세요.
4. 6절에서 존재하지 않는 도구 요청(`get_price`)이 들어왔을 때 프로그램이 예외로 죽지 않은 이유는 무엇인가요? 또한 실제 모델에게 없는 도구를 쓰라고 시켰을 때 에러 없이 답변이 끝났다면, 이를 무조건 안전한 성공이라고 볼 수 있을까요?
5. 7절에서 `recursion_limit` 값을 작게 줄이면 어떤 이점을 얻고, 반대로 어떤 한계가 생길 수 있나요?
6. 입력과 처리 순서가 항상 똑같이 정해져 있는 정형 업무(예: 매일 아침 같은 양식의 일일 매출 집계표 만들기)에 고정 workflow 대신 Agent를 쓰면 어떤 점이 불리해질까요?

## 정리

- 단일 모델 호출은 입력 하나에 텍스트 출력 하나를 낼 뿐이며, 도구를 묶어 주어도 모델은 **호출 요청(`tool_calls`)**만 작성합니다. 실제 함수 실행은 애플리케이션이 맡습니다.
- **Agent**는 모델 호출과 도구 실행 사이의 왕복을 실행 틀(harness)이 자동으로 이어 주는 구조입니다. 다음 행동을 모델이 매번 판단하므로 실행할 때마다 호출 횟수와 경로가 달라질 수 있습니다.
- 에이전트의 다섯 가지 구성요소 중 **모델, 도구, 지침, 상태**는 에이전트를 생성할 때 넘기고, **종료 조건(`recursion_limit`)**은 실행할 때 `config`로 지정합니다.
- 에이전트의 상태는 곧 **메시지 목록(`messages`)**입니다. 메시지 목록을 한 줄씩 읽을 수 있으면 에이전트가 내부에서 한 일을 정확히 설명할 수 있습니다.
- 없는 도구 호출은 `status='error'`인 `ToolMessage`로 모델에게 되돌아가고, 끝나지 않는 반복은 `recursion_limit`이 강제로 끊어 줍니다. 가장 경계해야 할 조용한 실패는 도구를 쓰지 않고 그럴듯하게 지어내어 답하는 경우입니다.

| 업무 성격 | 적합한 방식 (고정 workflow / Agent / 둘 다 아님) |
|---|---|
| 매일 정해진 쿼리로 같은 양식의 일일 보고서를 만든다 | **고정 workflow** (순서가 고정되어 있어 에이전트가 불필요함) |
| 고객의 다양한 문의를 읽고 필요한 사내 규정 문서를 찾아 답한다 | **Agent** (질문마다 찾아야 할 문서와 검색어가 달라짐) |
| 정해진 수식 한 줄로 끝나는 단순 산술 계산을 한다 | **둘 다 아님** (일반 파이썬 함수 하나로 충분함) |
| 사람의 확인 없이 고객 결제를 즉시 강제 취소한다 | **둘 다 아님** (되돌릴 수 없는 작업은 사람 승인 절차가 먼저 필요함) |

위 표는 내 업무에 Agent를 도입할 때 떠올려 볼 기준점입니다. 본인이 맡고 있는 업무 하나를 떠올려 위 표의 어느 칸에 해당하는지 생각해 보세요.

**호출 규모 안내:** 이 노트북의 모든 유료 셀은 `announce`가 실행 직전에 예상 호출 횟수를 먼저 알립니다. 위에서 아래로 모든 셀을 한 번씩 실행했을 때 `announce`에 적힌 `model_calls` 값을 모두 더하면 모델 호출은 총 30회 안팎으로 **추정**됩니다. 모델의 도구 선택에 따라 실행마다 횟수가 조금씩 달라질 수 있습니다.

**실측 기록:** 2026-11-02 실행 기록으로는 모든 셀을 위에서 아래로 한 번 실행했을 때 Claude 모델 호출이 29회, Decision API 호출은 0회였고 전체 실행 시간은 약 51.9초였습니다(입력 토큰 87,454개, 출력 토큰 2,685개). 이는 한 대의 컴퓨터에서 한 번 실행한 값이므로 위의 추정치(30회 안팎)를 대체하지 않으며, 모델의 도구 선택·네트워크 상태에 따라 달라질 수 있습니다.

다음 `02-Function-Calling-and-Tools.ipynb`에서는 도구의 이름, 설명, 인자 스키마가 모델의 선택을 어떻게 좌우하는지, 그리고 도구 오류가 났을 때 모델이 스스로 고치게 하려면 어떻게 설계해야 하는지 자세히 다룹니다.